# OTTO Multi-Objective Session-Based Recommendation - EDA

**Project 17, Independent Research.**

This notebook is a implementationed EDA skeleton. Cells are written but not executed. Run end-to-end after the OTTO dataset has been downloaded into `../data/` per the instructions in `../data/README.md`.

## Goals

1. Quantify the volume and shape of the dataset (sessions, events, items, time window).
2. Profile session length and event-type distribution.
3. Profile the item catalog (popularity tail, daily active items).
4. Surface temporal structure (hour-of-day, day-of-week effects).
5. Sanity-check label leakage (no future events in training context).
6. Produce a short list of preprocessing decisions for the modelling stage.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data')
pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (10, 4)

## 1. Load data

Prefer the parquet shards from `radek1/otto-full-optimized-memory-footprint` (~2.5 GB). Fall back to raw jsonl if necessary, but expect a 90+ GB peak RAM cost.

In [ ]:
train = pd.read_parquet(DATA / 'train.parquet')
test = pd.read_parquet(DATA / 'test.parquet')
print(f'train: {len(train):,} events, {train.session.nunique():,} sessions')
print(f'test:  {len(test):,} events, {test.session.nunique():,} sessions')

In [ ]:
train.head()

In [ ]:
train.dtypes

In [ ]:
train.describe(include='all')

In [ ]:
train.isna().sum()

## 2. Session-level statistics

Session length is a key driver of model design: a fat-tailed length distribution means the transformer needs an aggressive `max_seq_len` cap, and the baseline co-visitation needs a sliding-window prune.

In [ ]:
lens = train.groupby('session').size()
print(lens.describe())

In [ ]:
lens.clip(upper=50).hist(bins=50)
plt.xlabel('events per session (clipped at 50)')
plt.ylabel('count')
plt.title('Distribution of session length')
plt.show()

## 3. Event-type distribution

Clicks dominate orders by 100x or more. This explains why naive next-event models over-recommend popular items: the click signal swamps the order signal. The OTTO competition score weights orders 6x clicks for exactly this reason.

In [ ]:
train['type'].value_counts(normalize=True)

In [ ]:
train['type'].value_counts().plot.bar()
plt.title('Event type distribution (train)')
plt.ylabel('events')
plt.show()

## 4. Item popularity tail

Recsys catalogs are heavy-tailed by definition. Plotting log(rank) vs log(events per item) should yield a near-linear curve, which justifies cutting the embedding table at the top 200K-500K items without measurably hurting recall on the head.

In [ ]:
pop = train.aid.value_counts()
print(f'distinct aids: {len(pop):,}')
print(f'top 100K items cover {pop.head(100000).sum() / pop.sum():.1%} of events')
print(f'top 500K items cover {pop.head(500000).sum() / pop.sum():.1%} of events')

In [ ]:
fig, ax = plt.subplots()
ax.loglog(np.arange(1, len(pop) + 1), pop.values)
ax.set_xlabel('item rank')
ax.set_ylabel('events')
ax.set_title('Item popularity (log-log)')
plt.show()

## 5. Temporal structure

Day-of-week and hour-of-day effects matter for negative-sampling: a uniform random sample of items will over-represent items that are simply not on at the time of the target event. Position embeddings in SASRec absorb this only weakly, so the recommended fix is time-bucketed negatives.

In [ ]:
train['hour'] = pd.to_datetime(train['ts'], unit='ms').dt.hour
train['hour'].value_counts().sort_index().plot.bar()
plt.title('Event volume by hour of day')
plt.show()

## 6. Co-occurrence sanity

Quick spot check: how often is a click followed by a cart on the same item within 5 minutes? This is the click-to-cart conversion rate at the event level and gives an order-of-magnitude estimate of how predictable cart targets are.

In [ ]:
sample = train.groupby('session').head(20).copy()
sample = sample.sort_values(['session', 'ts'])
sample['next_aid'] = sample.groupby('session')['aid'].shift(-1)
sample['next_type'] = sample.groupby('session')['type'].shift(-1)
sample['next_dt_ms'] = sample.groupby('session')['ts'].shift(-1) - sample['ts']
click2cart = sample[(sample['type'] == 'clicks')
                    & (sample['next_type'] == 'carts')
                    & (sample['next_aid'] == sample['aid'])
                    & (sample['next_dt_ms'] <= 5 * 60 * 1000)]
print(f'{len(click2cart):,} click->cart same-item within 5 min')

## 7. Preprocessing decisions for modelling

1. **Item vocabulary cap.** Truncate to top 500K items if memory is tight, with all rarer items mapped to a shared OOV bucket. Otherwise keep the full 1.85M.
2. **Max sequence length.** Cap at 50 events; this covers the 99th percentile of sessions in the public stats.
3. **Padding strategy.** Left-pad with id 0 so the most recent event is always at the rightmost position, matching the causal-attention assumption.
4. **Negative sampling.** 200 uniform negatives per batch row. Time-bucketed negatives are deferred to v2.0.
5. **Validation split.** Hold out the last 7 days of train to mimic the public-private leaderboard split.
6. **Multi-task weighting.** Loss combines the three event types with the official competition weights (0.10 / 0.30 / 0.60).

Once these are confirmed, run `src/model_baseline.py` for the co-visitation reference, then `src/model_advanced.py` for the SASRec multi-task model.